# Расчет Funding Rate по методике СПБ биржи

## Формула Funding Rate

Сначала рассчитывается Premium Index:

$$
PI = \frac{MeanPrice - MeanIndex}{MeanIndex} \times 100\% \times Kpi
$$

где `MeanPrice` — средняя цена perpetual, а `MeanIndex` — средняя референсная цена.

Далее:

$$
FundingRate =
\frac{IR}{365}
- Clamp(PI, [-R1, R1])
+ Clamp(PI, [-R2, R2])
$$

Для 26.08.2026:

- R1 = 2%
- R2 = 0%
- IR = 0%
- Kpi = 1
  
## Источники

- [Механизм фандинга СПБ Биржи](https://spbexchange.ru/spbfuture/funding/)
- [Спецификации и коэффициенты СПБ Биржи](https://spbexchange.ru/about/raskrytie_informacii/vn_norm_docs/specf_cv/)

## 1. Загрузка данных

Используются секундные данные Binance по BTC и ETH.  

In [13]:
import pandas as pd
import numpy as np

btc = pd.read_csv("aggregated/btc_aggregated_1s.csv")
eth = pd.read_csv("aggregated/eth_aggregated_1s.csv")

btc["timestamp"] = pd.to_datetime(btc["timestamp"])
eth["timestamp"] = pd.to_datetime(eth["timestamp"])

print("BTC:", btc["timestamp"].min(), "—", btc["timestamp"].max())
print("ETH:", eth["timestamp"].min(), "—", eth["timestamp"].max())

BTC: 2026-08-26 14:00:11+00:00 — 2026-08-26 19:59:10+00:00
ETH: 2026-08-26 14:00:11+00:00 — 2026-08-26 19:59:10+00:00


## 2. Перевод в минуты

По методике СПБ биржи из секундных данных берётся последнее доступное наблюдение каждой минуты.

В расчёте:
- `binance_perp_mid` используется как цена perpetual;
- `binance_spot_mid` используется как референсная цена.

In [14]:
btc_minute = (
    btc.set_index("timestamp")[["binance_perp_mid", "binance_spot_mid"]]
    .resample("1min")
    .last()
    .dropna()
)

eth_minute = (
    eth.set_index("timestamp")[["binance_perp_mid", "binance_spot_mid"]]
    .resample("1min")
    .last()
    .dropna()
)

print("BTC:", len(btc_minute), "минут")
print("ETH:", len(eth_minute), "минут")

BTC: 360 минут
ETH: 360 минут


## 3. Средние цены

Для каждого актива рассчитываются:

- `MeanPrice` — средняя минутная цена perpetual Binance.
- `MeanReference` — средняя минутная spot-цена Binance.

$$
MeanPrice = \frac{\sum Price_i}{N}
$$

$$
MeanReference = \frac{\sum Reference_i}{N}
$$

In [15]:
btc_mean_price = btc_minute["binance_perp_mid"].mean()
btc_mean_reference = btc_minute["binance_spot_mid"].mean()

eth_mean_price = eth_minute["binance_perp_mid"].mean()
eth_mean_reference = eth_minute["binance_spot_mid"].mean()

print("BTC")
print("MeanPrice:", btc_mean_price)
print("MeanReference:", btc_mean_reference)

print("\nETH")
print("MeanPrice:", eth_mean_price)
print("MeanReference:", eth_mean_reference)

BTC
MeanPrice: 78216.94861111112
MeanReference: 78250.21552777778

ETH
MeanPrice: 2455.7431111111114
MeanReference: 2456.7801666666664


## 4. Premium Index

Premium Index показывает относительное отклонение средней цены perpetual от средней референсной цены.

$$
PI =
\frac{MeanPrice - MeanReference}{MeanReference}
\times 100\%
\times Kpi
$$

Для наших данных используется коэффициент:

$$
Kpi = 1
$$

In [16]:
Kpi = 1

btc_pi = (
    (btc_mean_price - btc_mean_reference)
    / btc_mean_reference
    * 100
    * Kpi
)

eth_pi = (
    (eth_mean_price - eth_mean_reference)
    / eth_mean_reference
    * 100
    * Kpi
)

print("BTC Premium Index:", btc_pi, "%")
print("ETH Premium Index:", eth_pi, "%")

BTC Premium Index: -0.042513514425849476 %
ETH Premium Index: -0.04221198012039217 %


## 5. Funding Rate

Funding Rate рассчитывается по формуле СПБ Биржи:

$$
FundingRate =
\frac{IR}{365}
-
Clamp(PI, [-R1, R1])
+
Clamp(PI, [-R2, R2])
$$

Для 26.08.2026:

- `R1 = 2%`
- `R2 = 0%`
- `IR = 0%`
- `Kpi = 1`

При этих параметрах формула упрощается до:

$$
FundingRate = -Clamp(PI, [-2\%, 2\%])
$$

In [17]:
R1 = 2.0
R2 = 0.0
IR = 0.0

btc_funding_rate = (
    IR / 365
    - np.clip(btc_pi, -R1, R1)
    + np.clip(btc_pi, -R2, R2)
)

eth_funding_rate = (
    IR / 365
    - np.clip(eth_pi, -R1, R1)
    + np.clip(eth_pi, -R2, R2)
)

print("BTC Funding Rate:", btc_funding_rate, "%")
print("ETH Funding Rate:", eth_funding_rate, "%")

BTC Funding Rate: 0.042513514425849476 %
ETH Funding Rate: 0.04221198012039217 %


## 6. Итоговый результат

Положительное значение Funding Rate значит, что по механизму СПБ Биржи выплату производит продавец (short), а покупатель (long) получает её.

В данном расчёте perpetual Binance в среднем торговался ниже spot-цены, поэтому Premium Index получился отрицательным, а Funding Rate — положительным.

In [18]:
result = pd.DataFrame({
    "Asset": ["BTC", "ETH"],
    "MeanPrice": [btc_mean_price, eth_mean_price],
    "MeanReference": [btc_mean_reference, eth_mean_reference],
    "PremiumIndex_%": [btc_pi, eth_pi],
    "FundingRate_%": [btc_funding_rate, eth_funding_rate]
})

result.round(6)

,Asset,MeanPrice,MeanReference,PremiumIndex_%,FundingRate_%
0,BTC,78216.948611,78250.215528,-0.042514,0.042514
1,ETH,2455.743111,2456.780167,-0.042212,0.042212
